# LC13 — A language model you can read (self-paced, ~45 min)

Before Period 2 puts an AI coding agent next to you, we build the smallest thing that deserves the name *language model*: a *bigram model*. It fits in one screen of Python, you can inspect every number in it, and it already has the one control knob (temperature) you will keep using in the real thing. By the end you will know exactly what a language model does — and exactly why the real ones need to be so much bigger.

No mathematics beyond counting and dividing. No machine-learning library. Just Python.

**How to use this notebook:** run the cells top to bottom, one at a time. Every code cell has a short note above it saying what the cell is for and what you should see when it runs — if what you see differs, stop there and ask your pod before moving on. Open it locally, from `course-material/notebooks/`, the way LC6 section 0 showed: the model trains on the course's own labs and guides, which it reads from the repository, so Colab will not find them.

## 0. Setup

`%pip install numpy --quiet` — numpy is the only package this notebook needs, and your course venv already has it, so this normally installs nothing. **What you see:** one line, "Note: you may need to restart the kernel…", or nothing at all — both fine, no restart needed.

In [1]:
# Install exactly what this notebook uses.
%pip install numpy --quiet

Note: you may need to restart the kernel to use updated packages.


The guard, as in every course notebook: `assert` stops the notebook right here, with the message, if it was not opened from the `notebooks/` folder of the course repository. **What you see:** no output at all — the good outcome.

In [2]:
from pathlib import Path
# Guard: this notebook expects to run from the notebooks/ folder of a clone of
# the course repository — the training text lives one level up in ../labs and
# ../guides. Failing here, early and clearly, beats a confusing error later.
assert Path("../labs").exists() and Path("../guides").exists(), (
    "Course folders not found. Clone KTH-EG2140/course-material and open "
    "this notebook from its notebooks/ folder.")

The imports. `re` is Python's regular-expression module, which you met in LC8's CIM parsing; `numpy` gives arrays you can divide in one go; `Counter` and `defaultdict` are two kinds of dictionary, explained in section 2 where they are used. **What you see:** nothing.

In [3]:
import re
from collections import Counter, defaultdict

import numpy as np

## 1. A corpus you have already read

A language model learns from text — a *corpus*. Ours: the course's own lab instructions and guides. That choice is deliberate: you know this text well, so you can judge the model's output the way you judged power-flow numbers in Period 1 — against knowledge you already have.

The cell below builds the corpus in two steps.

- **Read the prose.** Every `.md` file in `../labs` and `../guides`, sorted by name. `re.sub(pattern, " ", raw, flags=re.DOTALL)` replaces each fenced code block with a space: `re.DOTALL` lets `.` match newlines too, so one match can span a whole multi-line block, and the `?` in `.*?` makes it stop at the *first* closing fence rather than the last one in the file.
- **Cut it into words.** `re.findall(r"[a-zåäö]+", ...)` on the lowercased text returns every run of letters as a list. Anything else — digits, hyphens, dots — acts as a separator, so "EG2140" becomes the word `eg` and "N-1" becomes `n`. Real models cut text into *tokens*, pieces that are often smaller than words; we borrow the name for our list of words.

**What you see:** about 20 000 words of training text, about 2 300 of them distinct, then the first twelve — the opening of Lab 1, with `eg` where the course code was. The exact counts change whenever a lab or guide is edited; that is expected.

In [4]:
# Read every lab and guide, dropping the fenced code blocks (```...```) —
# we want the model to learn prose, not pip commands.
texts = []
for f in sorted(Path("../labs").glob("*.md")) + sorted(Path("../guides").glob("*.md")):
    raw = f.read_text(encoding="utf-8")
    prose = re.sub(r"```.*?```", " ", raw, flags=re.DOTALL)   # remove code fences
    texts.append(prose)
corpus_text = " ".join(texts)

# Tokenise: lowercase words only. Real models use smarter units ("tokens"),
# but words keep everything readable today.
tokens = re.findall(r"[a-zåäö]+", corpus_text.lower())
print(f"{len(tokens)} words of training text, {len(set(tokens))} distinct")
print("first twelve:", tokens[:12])

20215 words of training text, 2315 distinct
first twelve: ['lab', 'start', 'the', 'svedala', 'toolbox', 'from', 'notebook', 'to', 'package', 'eg', 'in', 'pairs']


## 2. Training = counting

A bigram model answers one question: *given the current word, what word comes next?* Training it means walking through the corpus once and counting every pair of neighbours. That is the entire "learning".

Three constructs you meet here for the first time:

- `Counter` is a dictionary that counts: `c[key] += 1` works even for a key it has never seen (it starts at 0), and `c.most_common(5)` returns the five largest counts as (word, count) pairs.
- `defaultdict(Counter)` is a dictionary that creates an empty `Counter` the first time a new key is used — so `follows[w][nxt] += 1` needs no "is this word new?" check.
- `zip(tokens, tokens[1:])` pairs the list with itself shifted by one: (word 1, word 2), (word 2, word 3), … — every pair of neighbours, once. In the print, `{word:12s}` pads the word to 12 characters so the counts line up.

**What you see:** "power" followed by `flow` about twenty times, then a few words seen once each. Read it as one row of a table: this is everything the model will ever know about what follows "power".

In [5]:
# For each word, count what follows it. This loop IS the training:
# one pass over the corpus, nothing but bookkeeping.
follows = defaultdict(Counter)
for w, nxt in zip(tokens, tokens[1:]):
    follows[w][nxt] += 1

# Inspect the model like any other table: what does it know about "power"?
print('After "power" the corpus continues with:')
for word, count in follows["power"].most_common(5):
    print(f"   {word:12s} seen {count} times")

After "power" the corpus continues with:
   flow         seen 21 times
   that         seen 1 times
   flows        seen 1 times
   system       seen 1 times


## 3. From counts to probabilities

Counts become a probability distribution by dividing each by the row's total. That table of rows — one distribution per word — **is the whole model**. Every "prediction" is just a lookup.

`next_distribution` turns one row of counts into probabilities. `np.array([...], dtype=float) / total` divides every count by the row total in one go — numpy applies the division to each element, no loop needed.

Below the function, `sorted(zip(words, probs), key=lambda t: -t[1])` sorts the (word, probability) pairs, largest probability first. `lambda t: -t[1]` is a one-line function without a name: it takes a pair `t` and returns minus its second element, and sorting by the negative puts the biggest first. `[:5]` keeps the top five; `{w!r}` prints the word in quotes and `{p:.3f}` with three decimals.

**What you see:** five continuations of "the", each between about 2 and 4 %. No word dominates: in this corpus "the" is followed by over 500 different words, which is what makes the word after "the" hard to predict.

In [6]:
def next_distribution(word):
    """The model's entire knowledge about one word: P(next | word)."""
    counts = follows[word]
    total = sum(counts.values())
    words = list(counts.keys())
    # Divide every count by the row total -> probabilities that sum to 1.
    probs = np.array([counts[w] for w in words], dtype=float) / total
    return words, probs

words, probs = next_distribution("the")
# Show the five most likely continuations of "the".
for w, p in sorted(zip(words, probs), key=lambda t: -t[1])[:5]:
    print(f'P({w!r} | "the") = {p:.3f}')

P('same' | "the") = 0.036
P('reference' | "the") = 0.027
P('first' | "the") = 0.019
P('course' | "the") = 0.019
P('test' | "the") = 0.019


## 4. Generation = sampling

To generate text, start from a word, draw the next one from its distribution, move there, repeat. Note what this means: the model never plans a sentence — each word is drawn looking at **one** word of history.

`generate` is the whole generation loop, in six lines:

- `np.random.default_rng(0)` makes a random-number generator with a fixed *seed* (0): the draws look random but are the same on every run from the top, so your text matches your neighbour's.
- `for _ in range(n_words):` repeats `n_words` times; `_` is the name Python programmers give a loop variable they never read. Inside, `rng.choice(words, p=p)` draws one word, each with its probability.
- The dead-end check stops early if the current word was never followed by anything (possible only for a word that occurs once, at the very end of the corpus). The two temperature lines are section 5's topic — at `temperature=1.0` they leave the probabilities unchanged. `" ".join(out)` glues the list into one string with spaces.

**What you see:** 26 words starting with "the" (the start word plus 25 draws) — grammatical in patches, meaningless as a whole. Running only this cell again gives a *different* string, because the generator has moved on; restart the kernel and run all to get the first one back.

In [7]:
rng = np.random.default_rng(0)   # fixed seed: same "random" text every run

def generate(start, n_words=25, temperature=1.0):
    """Sample a chain of words, one bigram step at a time."""
    out = [start]
    # Each pass: look up the current word's distribution, draw one successor.
    for _ in range(n_words):
        # Dead end: a word never followed by anything has no distribution - stop.
        if not follows[out[-1]]:
            break
        words, probs = next_distribution(out[-1])
        # Temperature reshapes the distribution before drawing (section 5).
        p = probs ** (1.0 / temperature)
        p = p / p.sum()
        out.append(rng.choice(words, p=p))
    return " ".join(out)

print(generate("the"))

the binary search you already at path needs it passed on canvas code gets a spike window here and run again same for review about seconds


Read that aloud. Every consecutive *pair* of words occurs somewhere in the course material — locally it sounds right — yet the sentence as a whole goes nowhere. Keep that observation; it becomes the punchline of section 6.

## 5. The temperature knob

Temperature rescales the probabilities before sampling: each probability is raised to the power $1/T$ and the row is re-normalised. Low $T$ exaggerates the differences — the most common continuation wins almost always. High $T$ flattens them — rare continuations get their chance. You will meet this exact parameter again in every AI API you touch in Period 2.

The next cell calls `generate` three times from the same start word, "your", once per temperature. `**` is Python's power operator, so `probs ** (1.0 / temperature)` in `generate` is exactly the $p^{1/T}$ above. **What you see:** three lines of 21 words each (the start word plus 20 draws), headed `T=0.3`, `T=1.0` and `T=2.0`.

In [8]:
# Same start word, same model — only the knob moves.
for T in (0.3, 1.0, 2.0):
    print(f"T={T}:")
    print("  ", generate("your", n_words=20, temperature=T))
    print()

T=0.3:
   your own location so the test that is the two lines are the reference solution the same table and the same

T=1.0:
   your toolbox screener output what reproducible environments survival git add svedala toolbox pipeline standardscaler the power flow does not commit the

T=2.0:
   your held out at idx column names its sitting reply with review about it take an undocumented one zone base p



Typical result: at `T=0.3` the model plays it safe and loops through the corpus's most-worn phrases; at `T=2.0` it free-associates into word salad; `T=1.0` sits between. Neither extreme is "wrong" — it is a dial you set per task. (Asking an agent for reproducible code edits and asking it to brainstorm test ideas want different settings.)

## 6. Why the real ones are so much bigger

Diagnosis time. Our model's failure is not the sampling — it is the **one word of memory**. Let's measure how little it takes before the model has, in effect, memorised the corpus instead of understanding it.

Two measurements in one cell:

- `[len(c) for c in follows.values()]` is a *list comprehension*: it builds a list with one number per word — how many different words followed it. `np.mean` averages that list, and `sum(1 for b in branching if b == 1)` counts the words that had exactly one continuation.
- The second half generates 15 more words and checks each adjacent pair against the model's own table. `(b in follows[a])` is `True` or `False`, and `sum` counts the `True`s — Lab 2's boolean-sum idiom.

**What you see:** between five and six continuations per word on average; roughly 1 000 of the 2 300 words with exactly **one** continuation — for those the model can only copy; and `15 of 15` generated pairs found in the corpus. That last number cannot come out any other way, because the model only ever draws a successor it has counted. That is the point, not a weak check.

In [9]:
# How many choices does the model actually have, on average?
# For each word: how many distinct successors did the corpus show it?
branching = [len(c) for c in follows.values()]
print(f"average continuations per word: {np.mean(branching):.1f}")
print(f"words with only ONE continuation: "
      f"{sum(1 for b in branching if b == 1)} of {len(branching)}")

# Check a generated pair-chain against the corpus: every adjacent pair the
# model produces was literally seen in training.
sample = generate("the", n_words=15).split()
pairs_in_corpus = sum((b in follows[a]) for a, b in zip(sample, sample[1:]))
print(f"generated pairs found verbatim in the corpus: "
      f"{pairs_in_corpus} of {len(sample)-1}")

average continuations per word: 5.6
words with only ONE continuation: 1029 of 2315
generated pairs found verbatim in the corpus: 15 of 15


Every pair it produces is a quotation; only the *chaining* is new. To do better, a model must look further back — and that is where counting collapses. With our vocabulary of about 2 300 words, a table over two-word histories would need about 5 million rows (2 300 × 2 300), three-word histories over 10 billion — and almost all of them would have **zero observations**. Counting cannot generalise to histories it never saw.

Modern language models break out of this with three moves you now have the vocabulary for:

- **Longer context, without a table** — instead of looking up the history, they *compute* with it: today's models weigh hundreds of thousands of tokens of history when predicting the next one (our model: one).
- **Meaning as numbers** — words become vectors, so "transformer" and "trafo" can share evidence instead of being unrelated rows. Unseen histories stop being dead ends.
- **Learned weights instead of counts** — billions of parameters tuned by gradient descent on trillions of tokens, so the "table" is replaced by a function that generalises.

But the loop you wrote today — *look at the context, produce a distribution over the next token, sample from it (with temperature), append, repeat* — is **exactly** the loop running inside the agents you will work with in Period 2. They are not a different kind of thing. They are this thing, scaled, with all the strengths and failure modes that follow: fluent locally, unanchored globally, tunable by the same knob you just turned. When an agent's confident code turns out subtly wrong in Lab 4 fashion — remember where the words come from.